# שיעור 04 - תבנית עיצוב שימוש בכלים

בשיעור זה תלמדו את תבנית העיצוב **שימוש בכלי** לסוכני בינה מלאכותית באמצעות מסגרת הסוכן של Microsoft (Python). נסקור:

- הגדרת כלים פונקציונליים עם העיטור `@tool` ופרמטרים עם טיפוסים
- מתן סכמות לכלים כך שהמודל ידע מה כל כלי עושה
- שליטה על ביצוע הכלים עם `approval_mode`
- החזרת **פלט מובנה** דרך מודלים של Pydantic ו-`response_format`

התסריט הוא **סוכן הזמנת נסיעות** שיכול לבדוק יעדים, לבדוק זמינות, ולהביא מידע על טיסות.


## התקנה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## הגדרת כלים עם דקורטור @tool

הדקורטור `@tool` הופך פונקציית פייתון פשוטה לכלי שהסוכן יכול לקרוא לו.  
נקודות מפתח:

- **מחרוזת התיעוד** הופכת לתיאור הכלי שהמודל רואה.  
- **הערות סוג** (כולל `Annotated` עם תיאורים) מגדירות את הסכימה של הכלי.  
- `approval_mode` שולט האם המשתמש חייב לאשר כל קריאה לפני ביצועה.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## יצירת סוכן עם כלים מרובים

שלח את כל שלושת הכלים ללקוח כך שהמודל יוכל להפעיל את אלו שהוא צריך כדי לענות על שאלת המשתמש.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## פלט מובנה עם כלים

על ידי הגדרת `response_format` למודל Pydantic, הסוכן מחויב להחזיר אובייקט JSON מטיפוס מוגדר במקום טקסט חופשי. זה שימושי כאשר קוד במעלה הזרם צריך לעבד את התוצאה בתכנות.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## תבניות אישור לכלים

הפרמטר `approval_mode` ב-`@tool` שולט האם קריאות לכלים דורשות אישור אנושי לפני ביצוע:

| מצב | התנהגות |
|---|---|
| `"never_require"` | הכלי רץ באופן אוטומטי — לא נדרש אישור משתמש. |
| `"always_require"` | כל קריאה חייבת לקבל אישור מהמשתמש לפני ביצועה. |

יש להשתמש ב-`"always_require"` עבור כלים שיש להם השפעות לוואי (כגון הזמנת טיסה, חיוב כרטיס אשראי) כדי שהאדם יישאר בלולאה.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## סיכום

בשיעור זה למדת כיצד:

1. **להגדיר כלים** באמצעות הדקורטור `@tool` עם פרמטרים עם טיפוסים וdocstrings המהווים את סכימת הכלי.
2. **לשלב מספר כלים** כך שהסוכן יוכל לקרוא להם ברצף כדי לענות על שאילתות מורכבות.
3. **להחזיר פלט מובנה** על ידי העברת מודל Pydantic כ-`response_format`.
4. **לשלוט באישור הכלי** באמצעות `approval_mode` כדי לשמור על מעורבות אנושית עבור פעולות רגישות.

תבניות אלה מהוות את היסוד לבניית סוכנים אמינים ומוכנים לפרודקשן שיכולים לתקשר בבטחה עם מערכות חיצוניות.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
